In [71]:
!pip install mne tensorflow scikit-learn

In [72]:
import numpy as np
import mne
import tensorflow as tf

from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, BatchNormalization,
    Activation, AveragePooling2D, Dropout, Permute,
    Reshape, Bidirectional, LSTM, Dense
)

from tensorflow.keras.models import Model

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [74]:
raw = mne.io.read_raw_gdf(file, preload=True)
raw.filter(8,30)

Extracting GDF parameters from /content/drive/MyDrive/BCI_IV_2b/B0101T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG:C3, EEG:Cz, EEG:C4, EOG:ch01, EOG:ch02, EOG:ch03
Creating raw.info structure...
Reading 0 ... 604802  =      0.000 ...  2419.208 secs...


/tmp/ipykernel_1391/3545072629.py:1: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(file, preload=True)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 s)



<RawGDF | B0101T.gdf, 6 x 604803 (2419.2 s), ~27.7 MiB, data loaded>

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [82]:
def EEGNet_BiLSTM(nb_classes=2,
                  Chans=3,
                  Samples=250,
                  dropoutRate=0.5,
                  kernLength=64,
                  F1=8,
                  D=2,
                  F2=16):

    input1 = Input(shape=(Chans, Samples, 1))

    # -----------------------------
    # EEGNet Block 1
    # -----------------------------
    block1 = Conv2D(F1, (1, kernLength),
                    padding='same',
                    use_bias=False)(input1)

    block1 = BatchNormalization()(block1)

    block1 = DepthwiseConv2D((Chans, 1),
                             use_bias=False,
                             depth_multiplier=D,
                             depthwise_constraint=tf.keras.constraints.max_norm(1.))(block1)

    block1 = BatchNormalization()(block1)

    block1 = Activation('elu')(block1)

    block1 = AveragePooling2D((1, 4))(block1)

    block1 = Dropout(dropoutRate)(block1)

    # -----------------------------
    # EEGNet Block 2
    # -----------------------------
    block2 = tf.keras.layers.SeparableConv2D(F2, (1, 16),
                             use_bias=False,
                             padding='same')(block1)

    block2 = BatchNormalization()(block2)

    block2 = Activation('elu')(block2)

    block2 = AveragePooling2D((1, 8))(block2)

    block2 = Dropout(dropoutRate)(block2)

    # -----------------------------
    # Reshape for LSTM
    # -----------------------------
    shape = block2.shape
    # (samples, 1, time_steps, features) -> (samples, time_steps, features)
    time_steps = shape[2]
    features = shape[1] * shape[3]

    x = Reshape((time_steps, features))(block2)

    # -----------------------------
    # BiLSTM Layer
    # -----------------------------
    x = Bidirectional(LSTM(64, return_sequences=False))(x)

    x = Dropout(0.5)(x)

    # -----------------------------
    # Classification
    # -----------------------------
    dense = Dense(nb_classes,
                  kernel_constraint=tf.keras.constraints.max_norm(0.25))(x)

    softmax = Activation('softmax')(dense)

    return Model(inputs=input1, outputs=softmax)

In [83]:
model = EEGNet_BiLSTM(
    nb_classes=2,
    Chans=X_train.shape[1],
    Samples=X_train.shape[2]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 3, 250, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 3, 250, 8)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 3, 250, 8)      │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_3              │ (None, 1, 250, 16)     │            48 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 1, 250, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_10 (Activation)      │ (None, 1, 250, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_6             │ (None, 1, 62, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 1, 62, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_2              │ (None, 1, 62, 16)      │           512 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 1, 62, 16)      │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_11 (Activation)      │ (None, 1, 62, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_7             │ (None, 1, 7, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 1, 7, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        41,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2)              │           258 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_12 (Activation)      │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,962 (167.82 KB)

 Trainable params: 42,882 (167.51 KB)

 Non-trainable params: 80 (320.00 B)

In [84]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 16s 68ms/step - accuracy: 0.5232 - loss: 0.6911 - val_accuracy: 0.5757 - val_loss: 0.6731
Epoch 2/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.6441 - loss: 0.6395 - val_accuracy: 0.6923 - val_loss: 0.5835
Epoch 3/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.6629 - loss: 0.6084 - val_accuracy: 0.7188 - val_loss: 0.5680
Epoch 4/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.6875 - loss: 0.5790 - val_accuracy: 0.7296 - val_loss: 0.5461
Epoch 5/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step - accuracy: 0.6765 - loss: 0.5979 - val_accuracy: 0.7031 - val_loss: 0.5673
Epoch 6/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - accuracy: 0.6971 - loss: 0.5698 - val_accuracy: 0.7296 - val_loss: 0.5390
Epoch 7/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 48ms/step - accuracy: 0.6901 - loss: 0.5719 - val_accuracy: 0.7380 - val_loss: 0.5269
Epoch 8/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - accuracy: 0.6870 - loss: 0.5714 -

In [85]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7136 - loss: 0.5443
Test Accuracy: 0.7105769515037537


In [86]:
y_pred = model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)
print(cm)

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step
[[395 125]
 [176 344]]


In [87]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [88]:
def EEGNet_BiLSTM(nb_classes=2,
                  Chans=3,
                  Samples=250,
                  dropoutRate=0.5,
                  kernLength=64,
                  F1=8,
                  D=2,
                  F2=16):

    input1 = Input(shape=(Chans, Samples, 1))

    # -----------------------------
    # EEGNet Block 1
    # -----------------------------
    block1 = Conv2D(F1, (1, kernLength),
                    padding='same',
                    use_bias=False)(input1)

    block1 = BatchNormalization()(block1)

    block1 = DepthwiseConv2D((Chans, 1),
                             use_bias=False,
                             depth_multiplier=D,
                             depthwise_constraint=tf.keras.constraints.max_norm(1.))(block1)

    block1 = BatchNormalization()(block1)

    block1 = Activation('elu')(block1)

    block1 = AveragePooling2D((1, 4))(block1)

    block1 = Dropout(dropoutRate)(block1)

    # -----------------------------
    # EEGNet Block 2
    # -----------------------------
    block2 = tf.keras.layers.SeparableConv2D(F2, (1, 16),
                             use_bias=False,
                             padding='same')(block1)

    block2 = BatchNormalization()(block2)

    block2 = Activation('elu')(block2)

    block2 = AveragePooling2D((1, 8))(block2)

    block2 = Dropout(dropoutRate)(block2)

    # -----------------------------
    # Reshape for LSTM
    # -----------------------------
    shape = block2.shape
    # (samples, 1, time_steps, features) -> (samples, time_steps, features)
    time_steps = shape[2]
    features = shape[1] * shape[3]

    x = Reshape((time_steps, features))(block2)

    # -----------------------------
    # BiLSTM Layer
    # -----------------------------
    x = Bidirectional(LSTM(64, return_sequences=False))(x)

    x = Dropout(0.5)(x)

    # -----------------------------
    # Classification
    # -----------------------------
    dense = Dense(nb_classes,
                  kernel_constraint=tf.keras.constraints.max_norm(0.25))(x)

    softmax = Activation('softmax')(dense)

    return Model(inputs=input1, outputs=softmax)

In [89]:
model = EEGNet_BiLSTM(
    nb_classes=2,
    Chans=X_train.shape[1],
    Samples=X_train.shape[2]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 3, 250, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 3, 250, 8)      │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 3, 250, 8)      │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_4              │ (None, 1, 250, 16)     │            48 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 1, 250, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_13 (Activation)      │ (None, 1, 250, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_8             │ (None, 1, 62, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 1, 62, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_3              │ (None, 1, 62, 16)      │           512 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 1, 62, 16)      │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_14 (Activation)      │ (None, 1, 62, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_9             │ (None, 1, 7, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 1, 7, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_4 (Reshape)             │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 128)            │        41,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 2)              │           258 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_15 (Activation)      │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,962 (167.82 KB)

 Trainable params: 42,882 (167.51 KB)

 Non-trainable params: 80 (320.00 B)

In [90]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - accuracy: 0.5090 - loss: 0.6922 - val_accuracy: 0.5673 - val_loss: 0.6775
Epoch 2/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.6583 - loss: 0.6300 - val_accuracy: 0.7067 - val_loss: 0.6208
Epoch 3/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.6675 - loss: 0.6038 - val_accuracy: 0.6983 - val_loss: 0.5755
Epoch 4/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.6809 - loss: 0.5973 - val_accuracy: 0.7031 - val_loss: 0.5772
Epoch 5/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 10s 45ms/step - accuracy: 0.6911 - loss: 0.5908 - val_accuracy: 0.7284 - val_loss: 0.5511
Epoch 6/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 11s 111ms/step - accuracy: 0.6922 - loss: 0.5848 - val_accuracy: 0.7332 - val_loss: 0.5391
Epoch 7/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 13s 123ms/step - accuracy: 0.7083 - loss: 0.5773 - val_accuracy: 0.7248 - val_loss: 0.5506
Epoch 8/100
104/104 ━━━━━━━━━━━━━━━━━━━━ 14s 60ms/step - accuracy: 0.7135 - loss: 0.

In [91]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7289 - loss: 0.5348
Test Accuracy: 0.7230769395828247


In [92]:
y_pred = model.predict(X_test)
y_pred = y_pred.argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)
print(cm)

33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step
[[385 135]
 [153 367]]


In [93]:
print(np.unique(y_train, return_counts=True))
print(np.unique(y_test, return_counts=True))

(array([0, 1]), array([2080, 2080]))
(array([0, 1]), array([520, 520]))
